# 🧪 POC — LLM Local no Google Colab (Ollama + Qwen3.5)
## Rodando Qwen 3.5 via Ollama, sem API externa!

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/poc_ollama_qwen35_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivo:** Provar que é possível rodar um modelo pequeno da família Qwen 3.5
direto no Google Colab gratuito (T4 GPU) usando **Ollama**, sem precisar de API key externa.

**Por que Ollama em vez de llama.cpp?**
- Ollama expõe uma API REST HTTP (compatível com OpenAI!) em `localhost:11434`
- Os alunos usam `requests.post()` — exatamente como fariam com a Gemini API
- **Tool calling nativo** — o modelo pode chamar ferramentas
- **Thinking mode** nativo — raciocínio interno antes da resposta
- Setup mais enxuto: um binário + um comando `ollama pull`

**Modelo padrão:** `qwen3.5:0.8b` (~1 GB) — rápido e leve, ideal pra aula.
Para mais capacidade, troque para `qwen3.5:2b` ou `qwen3.5:4b` nas células de inferência.

**Status:** POC experimental — não é material de aula (ainda).

---


## 1. Verificar GPU

⚠️ **Importante:** Vá em `Runtime → Change runtime type` e selecione **T4 GPU** antes de continuar.


In [ ]:
# Verificar GPU disponível
!nvidia-smi


## 2. Instalar e Configurar o Ollama

O Ollama é um servidor de inference local que roda como daemon. Vamos:
1. Instalar `zstd`, `lspci` e `lshw` (dependências que faltam no Colab)
2. Baixar e instalar o binário do Ollama
3. Corrigir o `LD_LIBRARY_PATH` (workaround pro Colab reconhecer a GPU)
4. Iniciar o servidor em background


In [ ]:
# Instalar dependências necessárias e o Ollama
# zstd: necessário pelo instalador do Ollama para descomprimir
# lspci, lshw: permitem que o Ollama detecte a GPU automaticamente
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

# Workaround: Ollama não detecta a GPU no Colab sem ajustar o LD_LIBRARY_PATH
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

print("✅ Ollama instalado!")

## 3. Iniciar o Servidor Ollama

O Ollama roda como um servidor em background na porta 11434.
Vamos iniciá-lo e verificar que está respondendo.


In [ ]:
# Iniciar Ollama em background
import subprocess
import time
import requests

# Matar qualquer instância anterior (seguro rodar múltiplas vezes)
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

# Iniciar servidor em background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env={**os.environ}
)

# Esperar o servidor ficar pronto (tentativas com timeout)
print("⏳ Aguardando Ollama iniciar...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama rodando na porta 11434!")
            break
    except:
        time.sleep(1)
else:
    print("❌ Ollama não iniciou em 30s. Tente rodar a célula novamente.")

## 4. Baixar Modelo e Criar Versão Otimizada

Vamos usar `ollama pull` para baixar o **Qwen3.5-0.8B** (~1 GB) e depois
criar um modelo customizado chamado `guilda-ia` com parâmetros otimizados.

**Por que um Modelfile?** Os defaults do Ollama (`temp=1.0`, `presence_penalty=1.5`)
causam overthinking no Qwen3.5. Nossos parâmetros são baseados no deploy local:
- `temperature: 1.0 → 0.7` — respostas mais focadas
- `presence_penalty: 1.5 → repeat_penalty 1.05` — sem loops de elaboração
- `top_p: 0.95 → 0.9` — menos verboso
- System prompt em português por padrão


In [ ]:
# Baixar o modelo Qwen 3.5
# Opções de tamanho (todas cabem na T4 gratuita):
#   qwen3.5:0.8b  → ~1.0 GB (mais rápido, recomendado pra aula)
#   qwen3.5:2b    → ~2.7 GB (bom equilíbrio)
#   qwen3.5:4b    → ~3.4 GB (mais qualidade, mais lento)
# Para trocar, basta mudar o modelo nas células seguintes.

MODEL = "guilda-ia"  # modelo customizado com parâmetros otimizados  # ← mude aqui para "qwen3.5:2b" ou "qwen3.5:4b"
!ollama pull {MODEL}

print(f"\n✅ Modelo {MODEL} baixado!")

## 5. Verificar Modelo e Teste Rápido (CLI)

Primeiro vamos listar os modelos baixados e testar via `ollama run` (direto no terminal).
Essa é a forma mais simples — o servidor já precisa estar rodando.


In [ ]:
# Listar modelos disponíveis
!ollama list

print("\n--- Teste rápido via CLI (modelo com parâmetros otimizados) ---")
!ollama run guilda-ia "Diga: Olá, Guilda de IA!" 2>/dev/null

print("\n--- Teste rápido via API ---")
import requests
r = requests.post("http://localhost:11434/api/chat", json={
    "model": "guilda-ia",
    "messages": [{"role": "user", "content": "Diga: Olá, Guilda de IA!"}],
    "stream": False
}, timeout=120)
print(f"🤖 {r.json()['message']['content']}")

## 6. Inferência via API REST (Python `requests`)

Aqui está o **ponto principal da POC**: o Ollama expõe uma API HTTP compatível com OpenAI.
Isso significa que o código Python é **idêntico** ao que usaríamos com a Gemini API ou OpenAI —
só muda a URL base de `https://generativelanguage.googleapis.com/...` pra `http://localhost:11434/...`.


In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/chat"

# Modelo padrão: mude aqui ou defina na célula 4
MODEL = "guilda-ia"  # modelo customizado com parâmetros otimizados

def chat(messages, model=None, stream=False):
    """Envia uma mensagem pro Ollama e retorna a resposta.

    Args:
        messages: lista de dicts no formato OpenAI [{"role": "...", "content": "..."}]
        model: nome do modelo no Ollama (padrão: qwen3.5:0.8b)
        stream: se True, retorna objeto response para iterar chunks

    Returns:
        dict com a resposta completa (ou response object se stream=True)
    """
    if model is None:
        model = MODEL
    payload = {
        "model": model,
        "messages": messages,
        "stream": stream
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=300)

    if response.status_code != 200:
        raise Exception(f"Erro {response.status_code}: {response.text}")

    if stream:
        return response
    return response.json()

# Teste simples
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
])

print(f"🤖 Resposta: {resultado['message']['content']}")
print(f"📊 Tokens: prompt={resultado.get('prompt_eval_count', '?')}, "
      f"completion={resultado.get('eval_count', '?')}")

## 7. Chat com Memória (Conversa Multi-turno)

A API do Ollama aceita o histórico completo de mensagens — igual à Gemini API.
Cada chamada envia toda a conversa, e o modelo mantém o contexto.


In [ ]:
# Chat com memória — acumulando o histórico
historico = [
    {"role": "system", "content": "Você é um assistente útil. Responda em português. Seja conciso."}
]

# Turno 1
historico.append({"role": "user", "content": "Meu nome é Lucas e eu ensino IA."})
r1 = chat(historico)
historico.append({"role": "assistant", "content": r1["message"]["content"]})
print(f"🤖: {r1['message']['content']}")

# Turno 2 — o modelo lembra do nome?
historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico)
historico.append({"role": "assistant", "content": r2["message"]["content"]})
print(f"🤖: {r2['message']['content']}")

## 8. Modo Thinking (Raciocínio Interno)

O Qwen 3.5 suporta **thinking mode** — o modelo raciocina internamente antes de responder.
Para ativar, basta incluir `/think` na mensagem ou usar o parâmetro `think: true` na API.


In [ ]:
# Thinking mode — o modelo raciocina antes de responder
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Quantos números primos existem entre 1 e 20? Resolva passo a passo. /think"}
])

# Ollama retorna thinking separado quando disponível
msg = resultado["message"]
thinking = msg.get("thinking", "(sem thinking)")
content = msg.get("content", "")

print(f"💭 Thinking:\n{thinking[:500]}...")
print(f"\n🤖 Resposta: {content}")

## 9. Streaming — Resposta em Tempo Real

Streaming mostra o texto conforme é gerado, igual ao ChatGPT.
Basta passar `stream=True` e iterar sobre os chunks.


In [ ]:
# Streaming de resposta
print("🤖 ", end="")

response = chat(
    [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Explique o que é uma API em 3 frases."}
    ],
    stream=True
)

for line in response.iter_lines():
    if line:
        chunk = json.loads(line)
        if "message" in chunk and "content" in chunk["message"]:
            text = chunk["message"]["content"]
            print(text, end="", flush=True)

print()  # newline no final

## 10. API Compatível com OpenAI

O Ollama também expõe um endpoint compatível com a OpenAI API em `/v1/chat/completions`.
Isso significa que **qualquer código que usa a OpenAI SDK pode apontar pro Ollama local**
trocando só a `base_url` e a `api_key`.

Isso é **muito relevante pra aula**: o padrão é o mesmo, só muda onde o request vai.


In [ ]:
# Usando o endpoint OpenAI-compatible
# Mesmo formato de request que a OpenAI API!

OPENAI_URL = "http://localhost:11434/v1/chat/completions"

payload = {
    "model": "guilda-ia",
    "messages": [
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "O que é Python em uma frase?"}
    ],
    "temperature": 0.7,
    "max_tokens": 64
}

response = requests.post(OPENAI_URL, json=payload, timeout=300)
data = response.json()

# Formato idêntico ao da OpenAI!
print(f"🤖 Resposta: {data['choices'][0]['message']['content']}")
print(f"📊 Model: {data['model']}")
print(f"📊 Usage: {data['usage']}")

## 11. Benchmark de Velocidade

Vamos medir quantos tokens por segundo o Qwen3.5-4B gera no Colab gratuito.


In [ ]:
import time

# Benchmark: gerar ~100 tokens e medir velocidade
prompt = "Conte uma história curta sobre um robô que aprende a cozinhar."

start = time.time()
resultado = chat([
    {"role": "user", "content": prompt}
])
elapsed = time.time() - start

eval_count = resultado.get("eval_count", 0)
prompt_count = resultado.get("prompt_eval_count", 0)
eval_duration = resultado.get("eval_duration", 0) / 1e9  # nanos → segundos

speed = eval_count / eval_duration if eval_duration > 0 else 0

print(f"⏱️ Tempo total: {elapsed:.1f}s")
print(f"📊 Tokens: prompt={prompt_count}, completion={eval_count}")
print(f"🚀 Velocidade: {speed:.1f} tokens/s")
print(f"\n📝 Resposta: {resultado['message']['content'][:200]}...")

## 12. 📋 Resultados da POC

### Modelo customizado `guilda-ia` 🎯

O modelo `guilda-ia` é criado via **Modelfile** com parâmetros otimizados para aula:

| Parâmetro | Ollama default | **Nosso ajuste** | Por quê |
|-----------|---------------|-------------------|---------|
| `temperature` | 1.0 | **0.7** | Menos aleatoriedade, respostas mais focadas |
| `top_p` | 0.95 | **0.9** | Reduz verbosidade |
| `top_k` | 20 | **20** | Mantido (já era bom) |
| `presence_penalty` | **1.5** | **1.05** (repeat_penalty) | 1.5 = overthinking severo! |

**O Ollama default `presence_penalty: 1.5` é o principal culpado do overthinking** — penaliza
qualquer repetição tão agressivamente que o modelo entra em loops de elaboração.

### Modelos base Qwen 3.5 (via Modelfile FROM)

| Modelo base | Download | VRAM ~ | Qualidade | Recomendação |
|-------------|----------|--------|-----------|-------------|
| **`qwen3.5:0.8b`** | **~1.0 GB** | **~2 GB** | Básica | **Padrão pra aula (rápido)** |
| `qwen3.5:2b` | ~2.7 GB | ~4 GB | Boa | Se quiser mais qualidade |
| `qwen3.5:4b` | ~3.4 GB | ~6 GB | Melhor | Para demonstrações avançadas |

Para trocar o modelo base, mude `FROM qwen3.5:0.8b` no Modelfile (célula 4) e recrie com `ollama create guilda-ia -f /tmp/Modelfile`.

### O que funciona ✅
- **Qwen3.5 roda no Colab gratuito** (T4 GPU, 16 GB VRAM) — inclusive o 0.8B!
- Modelo 0.8B é ultra-leve (~1 GB download, ~2 GB VRAM)
- **Modelfile com parâmetros otimizados** — sem overthinking, respostas concisas
- Ollama setup é mais enxuto que `llama-cpp-python` (sem compilação CUDA)
- **API REST HTTP** — alunos usam `requests.post()`, mesmo padrão da Gemini API
- **API compatível com OpenAI** (`/v1/chat/completions`) — fácil migração de código
- **Tool calling nativo** — o modelo pode chamar ferramentas
- **Thinking mode** — raciocínio interno separado da resposta
- Streaming funciona e melhora a experiência

### Por que Ollama é melhor que llama.cpp pra aula? 🎓

| Aspecto | `llama-cpp-python` (POC antiga) | **Ollama** (POC nova) |
|---------|-------------------------------|---------------------|
| Setup | Compila CUDA (~3-5 min) | Binário + pull (~2-3 min) |
| API | Python-only (`Llama.create_chat_completion()`) | **REST HTTP** (qualquer linguagem) |
| Tool calling | Não nativo | **Nativo** |
| Thinking | Via `extra_body` | **Nativo** (`/think`) |
| Compatibilidade | Binding específico | **OpenAI-compatible** |
| Didática | Ensina binding Python | **Ensina padrão HTTP** (transferível pra qualquer API) |
| Parâmetros | Hardcode no Python | **Modelfile** (reprodutível, versionável) |

### Limitações ⚠️
- **Setup demorado:** ~2-3 min por sessão Colab (instalar deps + Ollama + baixar modelo)
- **Sem persistência:** modelo baixado a cada sessão
- **Rate limits do Colab:** sessões gratuitas têm limite de horas
- **Workarounds:** `zstd`/`lspci`/`lshw` precisam ser instalados; `LD_LIBRARY_PATH` ajustado pro GPU

### Vabilidade para a Guilda
- **Aula 04 (Python + API):** Ollama como backend local — alunos fazem `requests.post()` sem API key
- **Aula futura (Agentes):** Tool calling nativo permite demonstrar agentes com ferramentas
- **Aula com hardware limitado:** `qwen3.5:0.8b` é ultra-leve, roda em qualquer GPU
- **Recomendação:** Usar Ollama local como complemento à Gemini API, mostrando o padrão HTTP unificado

---
*POC experimental — Guilda de IA UFVJM 2026.1*
